---
# `Maximal Marginal Relevance Retriever`
---

### Introduction
Based on Search Stragey, There are 3 types of Retrievers
1. Maximal Marginal Relevance Retriever (MMR)
2. Multi Query Retriever
3. Contextual Compression Retriever

### Maximal Marginal Relevance Retriever (MMR)
- In MMR , idea is that how can we pick results that are not only relevant to the query but also different from each other
- MMR is an information retrieval algo designed to reduce Redundacny/Duplicacy in the Retrieved data while maintaining High Relevance to the query.

### Why MMR Retriever
- In Regular simialarity search , Documents are going to be very similarr to each other, They are repeatly the same info, lacking diverse perspective
- MMR Retreiver avoids by picking the most relevance doc first.
- Then Picking the Next Most Relevant but least similar to already selected doc

# `Detailed Notes 1`

# Maximal Marginal Relevance (MMR) Retriever

**Maximal Marginal Relevance (MMR)** is a retrieval strategy used to return documents that are:

1. **Relevant to the user's query**
2. **Different from each other**

The key idea is:

> **Don't retrieve 5 documents that all say the same thing. Retrieve documents that are relevant while providing diverse information.**

---

## 1. Why Do We Need MMR?

Suppose the user asks:

> **"What are the benefits of Python?"**

Your vector database might return:

```text
Document 1 → Python is easy to learn.
Document 2 → Python has simple syntax.
Document 3 → Python is beginner-friendly.
Document 4 → Python is easy to understand.
Document 5 → Python has readable syntax.
```

All five documents are highly similar.

The problem:

```text
High relevance
      +
High redundancy
      ↓
Poor retrieval diversity
```

MMR tries to produce something like:

```text
Document 1 → Easy to learn
Document 2 → Large ecosystem
Document 3 → AI/ML libraries
Document 4 → Web development
Document 5 → Automation
```

Now the LLM gets **broader context**.

---

# 2. Similarity Search vs MMR

### Normal Similarity Search

It primarily asks:

> "Which documents are most similar to my query?"

```text
Query
  ↓
Vector Search
  ↓
Top K similar documents
```

Example:

```text
Query
 │
 ├── Doc A → 0.95
 ├── Doc B → 0.94
 ├── Doc C → 0.93
 ├── Doc D → 0.91
 └── Doc E → 0.90
```

But these documents might be almost identical.

---

### MMR

MMR asks:

> "Which documents are relevant to my query AND sufficiently different from the documents I've already selected?"

```text
Query
  ↓
Candidate Documents
  ↓
Relevance + Diversity
  ↓
MMR
  ↓
Diverse Relevant Documents
```

---

# 3. The MMR Formula

The core formula is:

[
MMR(D_i) =
\lambda \times Sim(D_i,Q)
-------------------------

(1-\lambda) \times \max_{D_j \in S} Sim(D_i,D_j)
]

Where:

* (D_i) = candidate document
* (Q) = user query
* (S) = documents already selected
* `Sim(Dᵢ, Q)` = similarity between document and query
* `Sim(Dᵢ, Dⱼ)` = similarity between candidate and already-selected documents
* (\lambda) = relevance vs diversity trade-off

The important part is:

```text
MMR =
Relevance
-
Redundancy
```

---

# 4. What Does Lambda Mean?

`lambda` controls the balance between **relevance** and **diversity**.

### λ close to 1

```text
More relevance
Less diversity
```

It behaves more like normal similarity search.

### λ close to 0

```text
Less relevance
More diversity
```

It prioritizes diversity more aggressively.

Conceptually:

```text
λ = 1.0
│
├── Relevance ██████████
└── Diversity

λ = 0.5
│
├── Relevance █████
└── Diversity █████

λ = 0.0
│
├── Relevance
└── Diversity ██████████
```

In practice, you usually choose a value somewhere in between rather than treating the extremes as useful production defaults.

---

# 5. How MMR Works Step-by-Step

Suppose we have:

```text
Query = "Benefits of Python"
```

Vector search produces candidates:

```text
D1 → highly relevant
D2 → highly relevant, almost same as D1
D3 → highly relevant, different topic
D4 → moderately relevant
D5 → highly relevant, almost same as D1
```

### Step 1 — Retrieve candidates

MMR first gets a larger candidate pool.

For example:

```text
fetch_k = 10
```

```text
Query
 ↓
Vector Store
 ↓
10 candidate documents
```

---

### Step 2 — Select the first document

The most relevant document is selected.

```text
Selected:
D1
```

---

### Step 3 — Evaluate remaining documents

Now MMR considers:

```text
D2
D3
D4
D5
...
```

It asks two questions:

```text
1. How relevant is this document to the query?
2. How similar is it to D1?
```

---

### Step 4 — Penalize redundancy

Suppose:

```text
D2:
Very relevant
But almost identical to D1
```

Its MMR score is reduced.

While:

```text
D3:
Very relevant
But provides different information
```

gets a better MMR score.

So:

```text
D1 → selected
D3 → selected
D4 → selected
...
```

---

# 6. Visual Example

Imagine documents positioned in embedding space:

```text
                 D2
              D1 ● ●
                 ● D5


       D3 ●


                         D4 ●

                    Query ●
```

Normal similarity search may return:

```text
D1
D2
D5
D3
D4
```

because D1, D2, and D5 are very similar.

MMR tries to avoid selecting too many documents from the same cluster:

```text
D1
D3
D4
...
```

So the final context covers more information.

---

# 7. MMR in LangChain

With a LangChain vector store, you can configure MMR as the search type.

For example:

```python
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)
```

Then:

```python
docs = retriever.invoke(
    "What are the benefits of Python?"
)
```

---

# 8. Important Parameters

### `search_type="mmr"`

Tells LangChain:

```text
Use Maximal Marginal Relevance
```

instead of ordinary similarity search.

---

### `k`

```python
"k": 4
```

Number of final documents returned.

```text
20 candidates
      ↓
     MMR
      ↓
4 final documents
```

---

### `fetch_k`

```python
"fetch_k": 20
```

Number of candidate documents initially retrieved before MMR selects the final `k`.

Think:

```text
fetch_k
   ↓
Candidate pool
   ↓
MMR
   ↓
k
   ↓
Final documents
```

Generally:

```text
fetch_k > k
```

is useful because MMR needs alternatives to choose from.

---

### `lambda_mult`

Controls:

```text
Relevance ←────────────→ Diversity
```

Higher:

```text
lambda_mult
      ↓
More relevance
Less diversity
```

Lower:

```text
lambda_mult
      ↓
More diversity
Less relevance
```

---

# 9. Complete Example

```code
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = Chroma(
    collection_name="documents",
    embedding_function=embeddings
)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

docs = retriever.invoke(
    "Explain the benefits of Python"
)

for doc in docs:
    print(doc.page_content)
```

The architecture is:

```text
User Query
    ↓
Embedding Model
    ↓
Vector Store
    ↓
fetch_k candidates
    ↓
MMR
    ↓
k diverse documents
    ↓
LLM
```

---

# 10. Why Is MMR Useful in RAG?

This is where MMR becomes particularly valuable.

Suppose your PDF contains 100 chunks about Transformers.

Normal similarity search might return:

```text
Chunk 10 → Self-attention
Chunk 11 → Self-attention
Chunk 12 → Self-attention
Chunk 13 → Self-attention
Chunk 14 → Self-attention
```

The LLM gets repetitive context.

MMR may retrieve:

```text
Chunk 10 → Self-attention
Chunk 27 → Multi-head attention
Chunk 43 → Positional encoding
Chunk 61 → Encoder-decoder architecture
Chunk 75 → Training
```

Now the context covers multiple aspects of the topic.

---

# 11. MMR Does NOT Mean "Retrieve Unrelated Documents"

This is an important misconception.

MMR is **not**:

```text
Find different documents regardless of relevance
```

It is:

```text
Find relevant documents
+
Avoid unnecessary duplication
```

So:

```text
Relevance is still important.
```

The objective is:

> **Maximum relevance with minimum redundancy.**

---

# 12. Similarity Search vs MMR

| Feature            | Similarity Search | MMR                           |
| ------------------ | ----------------- | ----------------------------- |
| Relevance          | High              | High                          |
| Diversity          | Low/variable      | Explicitly optimized          |
| Redundancy         | Can be high       | Reduced                       |
| Retrieval strategy | Similarity        | Similarity + diversity        |
| Good for           | Simple retrieval  | Diverse RAG context           |
| Main parameter     | `k`               | `k`, `fetch_k`, `lambda_mult` |

---

# 13. When Should You Use MMR?

MMR is particularly useful when:

### 1. Documents contain repetitive information

```text
Large documentation
Books
Research papers
FAQs
```

### 2. You want broad context

For example:

> "Explain Transformers."

You want information about:

```text
Attention
Positional Encoding
Encoder
Decoder
Training
Inference
```

rather than five chunks explaining attention.

### 3. Your chunks overlap heavily

If your text splitter creates overlapping chunks:

```text
Chunk 1
████████

Chunk 2
   ████████

Chunk 3
      ████████
```

normal similarity search may return highly overlapping chunks.

MMR can reduce this redundancy.

---

# 14. When MMR May Not Be Necessary

If your query requires **highly specific information**, diversity may not be particularly useful.

Example:

> "What is the exact timeout value configured for the payment API?"

You may want the most directly relevant chunks, even if they are semantically similar.

In such cases:

```text
Similarity Search
```

may be preferable.

---

# 15. MMR in the RAG Pipeline

Your complete understanding should now be:

```text
                    DOCUMENTS
                        ↓
                    Chunking
                        ↓
                   Embeddings
                        ↓
                  Vector Store
                        ↓
                 ┌─────────────┐
User Query ────→ │    MMR      │
                 └──────┬──────┘
                        ↓
             Relevant + Diverse Chunks
                        ↓
                     Prompt
                        ↓
                       LLM
                        ↓
                     Answer
```

### The key distinction

```code
Similarity Retriever:

"Give me the most similar documents."

MMR Retriever:

"Give me relevant documents,
but don't give me too many documents
that contain the same information."
```

**MMR = Relevance + Diversity − Redundancy.**
